In [1]:
!apt-get update && apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

!pkill ollama

!pip install -Uq crewai crewai-tools playwright pydantic pandas sentence-transformers pypdf faiss-cpu nest_asyncio
!playwright install chromium

!apt-get update -y
!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libdrm2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpangocairo-1.0-0 \
    libpango-1.0-0 \
    libgtk-3-0 \
    libnss3 \
    libnspr4 \
    libx11-xcb1 \
    libxcb1 \
    libxext6 \
    libx11-6

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,292 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,341 kB]

In [2]:
import re
import pandas as pd
import numpy as np
from typing import List, Optional, Dict
import time
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from crewai.tools import tool
from playwright.async_api import async_playwright
from urllib.parse import quote_plus
import asyncio
import time

import subprocess

try:
    subprocess.Popen(
        ["/usr/local/bin/ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("✅ Ollama server is starting...")
except FileNotFoundError:
    print("❌ Error: Ollama not found. Re-running installer...")
    !curl -fsSL https://ollama.com/install.sh | sh


time.sleep(15)


print("Pulling qwen2.5:14b ...")
!/usr/local/bin/ollama pull qwen2.5:14b

print("\n🚀 GPU is engaged and qwen2.5:14b  is ready for 281,880 rows!")

✅ Ollama server is starting...
Pulling qwen2.5:14b ...


🚀 GPU is engaged and qwen2.5:14b  is ready for 281,880 rows!


In [3]:
llm_model = LLM(
    model="ollama/qwen2.5:14b",
    base_url="http://localhost:11434",
    temperature=0,
    max_tokens=32000
)

In [4]:
from pydantic import BaseModel, Field
from typing import Optional, List


class Product(BaseModel):
    company: Optional[str] = Field(
        default=None,
        description="Brand or manufacturer of the product (e.g., Apple, Samsung)"
    )

    product: Optional[str] = Field(
        default=None,
        description="General product category (e.g., smartphone, laptop)"
    )

    model: Optional[str] = Field(
        default=None,
        description="Exact model name or number (e.g., iPhone 15 Pro, Galaxy S24)"
    )

    color: Optional[str] = Field(
        default=None,
        description="Color of the product if available"
    )

    price: Optional[float] = Field(
        default=None,
        ge=0,
        description="Price of the product"
    )

    location: Optional[str] = Field(
        default=None,
        description="Country or region where product is listed"
    )

    platform: Optional[str] = Field(
        default=None,
        description="E-commerce platform (Amazon, Noon, eBay, etc.)"
    )

    title: Optional[str] = Field(
        default=None,
        description="Full product title as shown on website"
    )

    rating: Optional[str] = Field(
        default=None,
        description="Customer rating (if available)"
    )

    product_url: Optional[str] = Field(
        default=None,
        description="Direct product link"
    )


class ProductList(BaseModel):
    products: List[Product]

In [5]:
products = ['mobile','smartphone','iphone','ps5','laptop']

websites = {
    "amazon": "https://www.amazon.com/s?k={}",
    "emax": "https://www.emaxme.com/search?q={}",
    "sharafdg": "https://uae.sharafdg.com/?s={}&post_type=product",
    "istyle": "https://istyle.ae/search?type=product%2Cpage&q={}",
    "Carrefour": "https://www.carrefouruae.com/mafuae/en/search?keyword={}",
    "Jumbo": "https://www.jumbo.ae/search/{}",
    "LULU" : "https://gcc.luluhypermarket.com/en-ae/list/?search_text={}",
    "Virgin" : "https://www.virginmegastore.ae/en/search/?text={}",
    "Jackys" : "https://www.jackyselectronics.com/catalogsearch/result/?q={}"
}

In [6]:
from urllib.parse import quote_plus

requests = []

for product in products:
    query = quote_plus(product)

    for site_name, template in websites.items():
        url = template.format(query)

        requests.append({
            "product": product,
            "site": site_name,
            "url": url
        })

In [7]:
print(requests)

[{'product': 'mobile', 'site': 'amazon', 'url': 'https://www.amazon.com/s?k=mobile'}, {'product': 'mobile', 'site': 'emax', 'url': 'https://www.emaxme.com/search?q=mobile'}, {'product': 'mobile', 'site': 'sharafdg', 'url': 'https://uae.sharafdg.com/?s=mobile&post_type=product'}, {'product': 'mobile', 'site': 'istyle', 'url': 'https://istyle.ae/search?type=product%2Cpage&q=mobile'}, {'product': 'mobile', 'site': 'Carrefour', 'url': 'https://www.carrefouruae.com/mafuae/en/search?keyword=mobile'}, {'product': 'mobile', 'site': 'Jumbo', 'url': 'https://www.jumbo.ae/search/mobile'}, {'product': 'mobile', 'site': 'LULU', 'url': 'https://gcc.luluhypermarket.com/en-ae/list/?search_text=mobile'}, {'product': 'mobile', 'site': 'Virgin', 'url': 'https://www.virginmegastore.ae/en/search/?text=mobile'}, {'product': 'mobile', 'site': 'Jackys', 'url': 'https://www.jackyselectronics.com/catalogsearch/result/?q=mobile'}, {'product': 'smartphone', 'site': 'amazon', 'url': 'https://www.amazon.com/s?k=sma

In [8]:
from crewai.tools import tool
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
import threading


@tool("HTML Scraper")
def html_scraper(url: str, product: str, site: str):
    """
    Playwright scraper with anti-bot settings and clean HTML output.
    """
    result = {}

    def run():
        with sync_playwright() as p:
            browser = p.chromium.launch(
                headless=True,
                args=[
                    "--no-sandbox",
                    "--disable-dev-shm-usage",
                    "--disable-blink-features=AutomationControlled"
                ]
            )

            context = browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                viewport={"width": 1280, "height": 800},
            )

            page = context.new_page()
            page.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            page.goto(url, wait_until="domcontentloaded", timeout=60000)
            page.wait_for_timeout(5000)

            html = page.content()


            blocked_signals = ["Access Denied", "403 Forbidden", "captcha", "blocked", "Robot or human"]
            if any(signal.lower() in html.lower() for signal in blocked_signals):
                result["html"] = None
                result["blocked"] = True
                browser.close()
                return


            soup = BeautifulSoup(html, "html.parser")
            for tag in soup(["script", "style", "nav", "footer", "header"]):
                tag.decompose()

            main = soup.select_one("main") or soup.select_one("#search") or soup.body
            result["html"] = str(main)
            result["blocked"] = False
            browser.close()

    thread = threading.Thread(target=run)
    thread.start()
    thread.join()


    if result.get("blocked"):
        return {
            "product": product,
            "site": site,
            "url": url,
            "html": None,
            "status": "BLOCKED"
        }

    return {
        "product": product,
        "site": site,
        "url": url,
        "html": result["html"],
        "status": "OK"
    }

In [9]:
def check_not_blocked(result):
    """
    Guardrail — reject the task output if site was blocked.
    """
    if result.get("status") == "BLOCKED":
        return (False, "Site blocked the request — skipping this task")

    if not result.get("html"):
        return (False, "No HTML returned — skipping this task")

    return (True, result)

In [10]:
from crewai import Agent

from crewai import Agent

html_agent = Agent(
    llm=llm_model,
    role="HTML Fetcher",
    goal="You MUST call html_scraper tool using the exact given inputs. Do not think. Do not generate anything else.",
    backstory="""
    You are a tool execution agent.

    RULES:
    - ALWAYS call html_scraper
    - NEVER generate URLs
    - NEVER modify inputs
    - NEVER answer without tool
    - If input is missing → FAIL (do not guess)
    """,
    tools=[html_scraper],
    verbose=True,
    allow_delegation=False,
    cache=False,
    max_iter=1
)

html_analyzer = Agent(
    llm=llm_model,
    role="HTML Analyzer",
    goal="""
    Extract structured product data ONLY from provided HTML blocks.

    OUTPUT RULE:
    - Return ONLY valid JSON
    - NO explanations
    - NO markdown
    - NO extra text
    """,
    backstory="""
    You are a strict and precise HTML parser for ecommerce data.

    CRITICAL RULES:
    - ONLY use provided HTML
    - DO NOT use prior knowledge
    - DO NOT guess missing values
    - If a field is not present → return null
    - DO NOT hallucinate products, prices, or ratings
    - Extract ALL products found in HTML

    You must extract:
    - company (brand/manufacturer if visible)
    - product (category/type)
    - model (exact model name if visible)
    - color (if explicitly shown)
    - price
    - location (only if explicitly present)
    - platform (Amazon, Noon, eBay, etc.)
    - title
    - rating
    - product_url
    """,
    verbose=True,
    allow_delegation=False
)

In [ ]:
import json
from crewai import Crew, Task

results = []

for req in requests:

    extract_task = Task(
        description=f"""
        Call html_scraper with EXACT parameters:

        url="{req['url']}"
        product="{req['product']}"
        site="{req['site']}"

        RULES:
        - DO NOT modify URL
        - DO NOT guess
        - JUST call tool
        """,
        expected_output="Raw HTML",
        agent=html_agent
    )


    analyze_task = Task(
    description="""
You will receive HTML blocks from an ecommerce website.

Extract ALL product information.

OUTPUT FORMAT (STRICT JSON ONLY):

{
  "products": [
    {
      "company": null,
      "product": null,
      "model": null,
      "color": null,
      "price": null,
      "location": null,
      "platform": null,
      "title": null,
      "rating": null,
      "product_url": null
    }
  ]
}

RULES:
- Extract ONLY what is visible in HTML
- Do NOT guess missing fields
- If not found → null
- DO NOT return explanations
""",
    agent=html_analyzer,
    context=[extract_task],
    expected_output="JSON matching Product Pydantic schema"
)


    crew = Crew(
        agents=[html_agent, html_analyzer],
        tasks=[extract_task, analyze_task],
        verbose=True
    )

    try:
        result = crew.kickoff()


        raw_output = result.raw if hasattr(result, "raw") else str(result)

        data = json.loads(raw_output)
        validated = ProductList.model_validate(data)

        results.append({
            "product": req["product"],
            "site": req["site"],
            "url": req["url"],
            "output": validated.model_dump()
        })


        for item in validated.products:
            all_rows.append({
                "search_product": req["product"],
                "site": req["site"],
                "url": req["url"],

                "company": item.company,
                "product": item.product,
                "model": item.model,
                "color": item.color,
                "price": item.price,
                "location": item.location,
                "platform": item.platform,
                "title": item.title,
                "rating": item.rating,
                "product_url": item.product_url
            })

        print(f"✅ Done: {req['site']} - {req['product']}")

    except Exception as e:

        print(f"❌ Failed: {req['site']} - {req['product']} | {str(e)}")

        results.append({
            "product": req["product"],
            "site": req["site"],
            "url": req["url"],
            "output": None,
            "error": str(e)
        })




df = pd.DataFrame(all_rows)


df = df.dropna(subset=["title"])


df["price"] = pd.to_numeric(df["price"], errors="coerce")

df.to_csv("products.csv", index=False)

print("\n🎉 CSV exported successfully: products.csv")
print(df.head())

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c172f210-c8bb-46b2-b34d-bf5d8433b762                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Call html_scraper with EXACT parameters:                                                               │
│                                                                                                                 │
│          url="https://www.amazon.com/s?k=mobile"                                                                │
│          product="mobile"                                                                                       │
│          site="amazon"                                                                                          │
│                                                                                                                 │
│          RULES:                                                                                                 │
│          - DO NOT modify URL                                                                                    │
│          - DO NOT guess                                                                                         │
│          - JUST call tool                                                                                       │
│                                                                                                                 │
│  ID: 81610935-bef8-4208-a29f-bc91a5373b6f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HTML Fetcher                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Call html_scraper with EXACT parameters:                                                               │
│                                                                                                                 │
│          url="https://www.amazon.com/s?k=mobile"                                                                │
│          product="mobile"                                                                                       │
│          site="amazon"                                                                                          │
│                                                                                                                 │
│          RULES:                                                                                                 │
│          - DO NOT modify URL                                                                                    │
│          - DO NOT guess                                                                                         │
│          - JUST call tool                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: html_scraper                                                                                             │
│  Args: {'product': 'mobile', 'site': 'amazon', 'url': 'https://www.amazon.com/s?k=mobile'}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool html_scraper executed with result: {'product': 'mobile', 'site': 'amazon', 'url': 'https://www.amazon.com/s?k=mobile', 'html': '<body>\n<p>\xa0</p>\n<center><img alt="website temporarily unavailable" height="300" src="http://g-ecx.imag...
Maximum iterations reached. Requesting final answer.


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: html_scraper                                                                                             │
│  Output: {'product': 'mobile', 'site': 'amazon', 'url': 'https://www.amazon.com/s?k=mobile', 'html':            │
│  '<body>\n<p>\xa0</p>\n<center><img alt="website temporarily unavailable" height="300"                          │
│  src="http://g-ecx.images-amazon.com/images/G/01/website/errors/503/generic.png"                                │
│  width="500"/></center>\n</body>', 'status': 'OK'}                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HTML Fetcher                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```html                                                                                                        │
│  <body>                                                                                                         │
│  <p>&nbsp;</p>                                                                                                  │
│  <center><img alt="website temporarily unavailable" height="300"                                                │
│  src="http://g-ecx.images-amazon.com/images/G/01/website/errors/503/generic.png" width="500"/></center>         │
│  </body>                                                                                                        │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Call html_scraper with EXACT parameters:                                                               │
│                                                                                                                 │
│          url="https://www.amazon.com/s?k=mobile"                                                                │
│          product="mobile"                                                                                       │
│          site="amazon"                                                                                          │
│                                                                                                                 │
│          RULES:                                                                                                 │
│          - DO NOT modify URL                                                                                    │
│          - DO NOT guess                                                                                         │
│          - JUST call tool                                                                                       │
│                                                                                                                 │
│  Agent: HTML Fetcher                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  You will receive HTML blocks from an ecommerce website.                                                        │
│                                                                                                                 │
│  Extract ALL product information.                                                                               │
│                                                                                                                 │
│  OUTPUT FORMAT (STRICT JSON ONLY):                                                                              │
│                                                                                                                 │
│  {                                                                                                              │
│    "products": [                                                                                                │
│      {                                                                                                          │
│        "company": null,                                                                                         │
│        "product": null,                                                                                         │
│        "model": null,                                                                                           │
│        "color": null,                                                                                           │
│        "price": null,                                                                                           │
│        "location": null,                                                                                        │
│        "platform": null,                                                                                        │
│        "title": null,                                                                                           │
│        "rating": null,                                                                                          │
│        "product_url": null                                                                                      │
│      }                                                                                                          │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
│  RULES:                                                                                                         │
│  - Extract ONLY what is visible in HTML                                                                         │
│  - Do NOT guess missing fields                                                                                  │
│  - If not found → null                                                                                          │
│  - DO NOT return explanations                                                                                   │
│                                                                                                                 │
│  ID: ff1587cf-961d-4df0-8cf2-a6f3a36e93d5                                                                       │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HTML Analyzer                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You will receive HTML blocks from an ecommerce website.                                                        │
│                                                                                                                 │
│  Extract ALL product information.                                                                               │
│                                                                                                                 │
│  OUTPUT FORMAT (STRICT JSON ONLY):                                                                              │
│                                                                                                                 │
│  {                                                                                                              │
│    "products": [                                                                                                │
│      {                                                                                                          │
│        "company": null,                                                                                         │
│        "product": null,                                                                                         │
│        "model": null,                                                                                           │
│        "color": null,                                                                                           │
│        "price": null,                                                                                           │
│        "location": null,                                                                                        │
│        "platform": null,                                                                                        │
│        "title": null,                                                                                           │
│        "rating": null,                                                                                          │
│        "product_url": null                                                                                      │
│      }                                                                                                          │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
│  RULES:                                                                                                         │
│  - Extract ONLY what is visible in HTML                                                                         │
│  - Do NOT guess missing fields                                                                                  │
│  - If not found → null                                                                                          │
│  - DO NOT return explanations                                                                                   │
│                                                                                                                 │
│                                                        